# Proyecto final · 03 · Gold y calidad

**Componente 1 · Sección 3 del examen.** Una tabla o view gold que responda una pregunta de negocio, y reglas de calidad que se cumplan.

Lo que se evalúa:
- Al menos un objeto gold con una pregunta de negocio clara.
- Al menos **dos** reglas de calidad que Databricks haga cumplir: `NOT NULL`, `CHECK`, o expectations si usas pipelines declarativos.
- La evidencia de que una fila mala **se rechaza**: el error literal en la salida de una celda.

Recuerda: una materialized view no se crea desde un notebook en serverless. Si quieres una, créala en el editor SQL y deja aquí la consulta que la lee.

In [0]:
-- Seleccionar el catálogo del proyecto para crear y consultar la capa Gold.
USE CATALOG pf_fallas;

In [0]:
-- Crear la tabla Gold con fabricante obligatorio.
-- Después, agregar una regla CHECK que rechace cantidades iguales o menores a cero.
CREATE OR REPLACE TABLE pf_fallas.gold.vehiculos_por_fabricante (
    make STRING NOT NULL,
    type_code STRING,
    total_vehiculos BIGINT,
    porcentaje_total DECIMAL(7, 2)
)
USING DELTA;

ALTER TABLE pf_fallas.gold.vehiculos_por_fabricante
ADD CONSTRAINT cantidad_positiva CHECK (total_vehiculos > 0);

In [0]:
-- Cargar el resumen de Silver en Gold.
-- Sobrescribir los resultados para evitar duplicarlos al ejecutar nuevamente.
INSERT OVERWRITE TABLE gold.vehiculos_por_fabricante
SELECT
    make,
    type_code,
    COUNT(*) AS total_vehiculos,
    CAST(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER ()
        AS DECIMAL(7, 2)
    ) AS porcentaje_total
FROM silver.vehiculos
GROUP BY make, type_code;

num_affected_rows,num_inserted_rows
65,65


In [0]:
-- Mostrar los fabricantes y tipos con más vehículos.
-- Este resultado responde la pregunta de negocio del proyecto.
SELECT *
FROM gold.vehiculos_por_fabricante
ORDER BY total_vehiculos DESC, make, type_code;

make,type_code,total_vehiculos,porcentaje_total
TESLA,BEV,12396,41.32
NISSAN,BEV,2074,6.91
CHEVROLET,BEV,1754,5.85
TOYOTA,PHEV,1124,3.75
FORD,BEV,1020,3.40
KIA,BEV,918,3.06
HYUNDAI,BEV,907,3.02
RIVIAN,BEV,831,2.77
VOLKSWAGEN,BEV,746,2.49
BMW,PHEV,740,2.47


In [0]:
-- Mostrar el CHECK en las propiedades de la tabla.
SHOW TBLPROPERTIES pf_fallas.gold.vehiculos_por_fabricante;

key,value
delta.constraints.cantidad_positiva,total_vehiculos > 0
delta.enableDeletionVectors,true
delta.enableRowTracking,true
delta.feature.appendOnly,supported
delta.feature.checkConstraints,supported
delta.feature.deletionVectors,supported
delta.feature.domainMetadata,supported
delta.feature.invariants,supported
delta.feature.rowTracking,supported
delta.minReaderVersion,3


In [0]:
-- Revisar la definición de la tabla para comprobar el NOT NULL de make.
SHOW CREATE TABLE pf_fallas.gold.vehiculos_por_fabricante;

createtab_stmt
"CREATE TABLE pf_fallas.gold.vehiculos_por_fabricante ( make STRING COLLATE UTF8_BINARY NOT NULL, type_code STRING COLLATE UTF8_BINARY, total_vehiculos BIGINT, porcentaje_total DECIMAL(7,2)) USING delta DEFAULT COLLATION UTF8_BINARY TBLPROPERTIES ( 'delta.constraints.cantidad_positiva' = 'total_vehiculos > 0', 'delta.enableDeletionVectors' = 'true', 'delta.enableRowTracking' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.checkConstraints' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.domainMetadata' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.feature.rowTracking' = 'supported', 'delta.minReaderVersion' = '3', 'delta.minWriterVersion' = '7', 'delta.parquet.compression.codec' = 'zstd', 'delta.parquet.format.version' = '2.12.0', 'delta.parquet.format.version.afe.internal' = '2.12.0')"


### Evidencia: una fila que se rechaza
Escribe un `INSERT` con una fila que rompa tu regla y córrelo a mano. **La celda tiene que fallar**: ese error es la evidencia. Déjala fallada en la entrega, y no la incluyas en el job.

In [0]:
-- Prueba manual: intentar insertar una cantidad negativa.
-- Esta celda debe fallar por la restricción cantidad_positiva.
-- Conservar el error como evidencia y excluir esta prueba del Job.
INSERT INTO pf_fallas.gold.vehiculos_por_fabricante
    (make, type_code, total_vehiculos, porcentaje_total)
VALUES
    ('PRUEBA_CALIDAD', 'BEV', -1, 0.00);

---------------------------------------------------------------------------
SparkConnectGrpcException                 Traceback (most recent call last)
File <command-7748707505045051>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "-- Prueba manual: intentar insertar una cantidad negativa.\n-- Esta celda debe fallar por la restricción cantidad_positiva.\n-- Conservar el error como evidencia y excluir esta prueba del Job.\nINSERT INTO pf_fallas.gold.vehiculos_por_fabricante\n    (make, type_code, total_vehiculos, porcentaje_total)\nVALUES\n    ('PRUEBA_CALIDAD', 'BEV', -1, 0.00);\n")

File /databricks/python/lib/python3.11/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silence

### TU RESPUESTA
¿Qué pregunta de negocio responde tu tabla gold? ¿Por qué elegiste tabla, view o materialized view?

La tabla Gold responde cuáles fabricantes concentran más vehículos y cómo
se distribuyen entre eléctricos puros (`BEV`) e híbridos enchufables (`PHEV`).

Cada fila representa una combinación de fabricante y tipo de vehículo.
Incluye la cantidad de vehículos y su porcentaje sobre el total del dataset.

Se eligió una tabla Delta para guardar el resumen y aplicar reglas de calidad:
`make` no permite nulos y `total_vehiculos` debe ser mayor que cero.

La prueba con una cantidad de `-1` produjo el error
`DELTA_VIOLATE_CONSTRAINT_WITH_VALUES`, confirmando que Databricks rechazó
la fila incorrecta.

In [0]:
-- Verificar cuántos grupos y fabricantes contiene Gold,
-- y confirmar que el resumen conserva los 30.000 vehículos.
SELECT
    COUNT(*) AS grupos_fabricante_tipo,
    COUNT(DISTINCT make) AS fabricantes,
    SUM(total_vehiculos) AS total_vehiculos
FROM pf_fallas.gold.vehiculos_por_fabricante;

grupos_fabricante_tipo,fabricantes,total_vehiculos
65,44,30000
